# Store——跨会话存储

Checkpointer 保存的是对话状态（消息历史），对话结束就没有了。Store 是一种跨会话的持久化存储，用于保存用户偏好、配置信息等。

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

环境变量已加载


## Checkpointer vs Store

| 维度 | Checkpointer | Store |
| --- | --- | --- |
| 存储内容 | 对话状态（消息历史） | 应用数据（偏好、配置） |
| 生命周期 | 对话结束后可丢弃 | 长期持久化 |
| 访问方式 | Agent 自动管理 | 工具通过 InjectedStore 访问 |
| 适用场景 | 多轮对话记忆 | 用户偏好、知识库、业务数据 |

## 使用 Store

创建 Store，传给 `create_agent()`，工具通过 `InjectedStore` 读取。

In [2]:
from typing import Annotated
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, InjectedStore
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

# 创建 Store 并预置数据
store = InMemoryStore()
store.put(("runoob", "courses"), "catalog", {
    "data": {
        "Python3 基础教程": {"price": "免费", "duration": "20小时"},
        "Python 数据分析": {"price": "会员", "duration": "30小时"},
    }
})

@tool
def query_course_price(course_name: str, store: Annotated[BaseStore, InjectedStore()]) -> str:
    """查询课程价格"""
    item = store.get(("runoob", "courses"), "catalog")
    catalog = item.value["data"] if item else {}
    if course_name in catalog:
        info = catalog[course_name]
        return f"《{course_name}》：{info['price']}，{info['duration']}"
    return f"未找到《{course_name}》"

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, tools=[query_course_price], store=store, system_prompt="你是课程顾问。")

result = agent.invoke({"messages": [HumanMessage(content="Python3 基础教程多少钱？")]})
print(result["messages"][-1].content)

好消息！🎉 **《Python3 基础教程》** 是 **免费** 的！课程时长 **20小时**，内容非常丰富，完全不需要花钱就能学习哦！

如果你想进一步了解课程大纲或者有其他编程课程想要咨询，随时问我！😊


**术语：**

- **Store**：跨会话持久化存储
- **InjectedStore**：注入 Store 到工具的机制
- **InMemoryStore**：内存存储实现，测试用
- **Namespace**：Store 的命名空间，用于数据隔离